In [ ]:
import pandas as pd

DIR1 = r'../data/clean/stock_prices.csv'
data = pd.read_csv(DIR1, index_col=0)
print(data.head())

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from src.processing import set_plot_font, save_chart
import seaborn as sns

plt.figure(figsize=(8,6))
set_plot_font()
ax = sns.scatterplot(data=data, x='z/s', y='std', hue='label', palette='grey', style = 'label', markers=['v', 'o', 'D'], s=100, linewidth=.6, edgecolor='black', alpha=0.9)
sns.move_legend(ax, "upper right", title=None)
ax.xaxis.set_major_formatter(ticker.PercentFormatter(1.0))
plt.xlabel('Stopa zwrotu')
plt.ylabel('Odchylenie standardowe')
save_chart('stopa zwrotu std')

In [ ]:
import pandas as pd

DIR_1 = r'../data/clean/a_value.csv'
DIR_2 = r'../data/clean/b_value.csv'
DIR_3 = r'../data/clean/c_value.csv'
DIR_4 = r'../data/clean/wig.csv'
a = pd.read_csv(DIR_1, index_col=0)['value']
a.name = 'A'
b = pd.read_csv(DIR_2, index_col=0)['value']
b.name = 'B'
c = pd.read_csv(DIR_3, index_col=0)['value']
c.name = 'C'
wig=pd.read_csv(DIR_4, index_col=0)['Zamkniecie']
wig.name = 'WIG'

df = pd.DataFrame([a, b, c, wig]).transpose()
values = pd.DataFrame()
values[['A','B','C']] = df[['A','B','C']]/100000-1
values['WIG']=df['WIG']/df.loc['2025-01-02', 'WIG']-1
values.index=pd.DatetimeIndex(values.index)
print(values.index)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from src.processing import set_plot_font, save_chart
from matplotlib import ticker

colors = sns.color_palette('mako', n_colors=3)
fig, ax = plt.subplots(figsize=(10,6))
set_plot_font()
ax.plot(values.index, values['WIG'], c='black', linewidth=2, linestyle='--', label='WIG')
ax.plot(values.index, values['A'], c=colors[0], label='A')
ax.plot(values.index, values['B'], c=colors[1], label='B')
ax.plot(values.index, values['C'], c=colors[2], label='C')

ax.legend(frameon=True)
plt.xlabel('')
plt.ylabel('Zmiana procentowa')
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%m.%Y"))
ax.yaxis.set_major_formatter(ticker.PercentFormatter(xmax=1))
ax.axhline(y=0, c='grey', linestyle='--', linewidth=1)
ax.yaxis.grid(True, linestyle=':', alpha=0.7)
#save_chart('zwrot_portfele')

In [ ]:
import numpy as np

dywidenda = pd.read_csv(r'../data/clean/dywidenda.csv', index_col = 0)
dywidenda.columns=['dywidenda']
labeled_data = pd.read_csv(r'../data/clean/labeled_data.csv', index_col = 0)['label']
dywidenda = dywidenda.merge(labeled_data, left_index=True, right_index=True)
dywidenda['procent'] = dywidenda['dywidenda']/100000

fig, ax = plt.subplots(figsize=(8,6))
set_plot_font()
sns.barplot(dywidenda, ax = ax, x = 'label', y='procent', estimator='sum', errorbar=None, color='#696969', linewidth=.5, edgecolor='black')
plt.xlabel('')
plt.ylabel('Stopa zwrotu z dywidend')
ax.yaxis.set_major_formatter(ticker.PercentFormatter(1))
sns.despine()
#save_chart('bar_dywidendy')

In [ ]:
from src.indicators import get_share_price
stopa_wolna = get_share_price('10YPLY.B').query('rok==2025')['Zamkniecie'].mean()
sr = df/df.shift()-1

sharp = ((sr.mean()*249-stopa_wolna/100)/(sr.std()*np.sqrt(249))).round(2)
sharp

In [ ]:
from matplotlib import ticker
from matplotlib.dates import MonthLocator
fig, ax = plt.subplots(2, 2, sharey=True, figsize=(8,8))
pairs =np.array([
        [0, 0, 0],
        [0, 1, 1],
        [1, 0, 2],
        [1, 1, 3]
])

for i, j, p in pairs:
    ax[i, j].plot(sr.index, sr.iloc[:, p], linewidth=.95, color='#636363')
    ax[i, j].set_title(sr.iloc[:, p].name, fontsize=10)
    ax[i, j].axhline(0, linewidth=.7, alpha=.8, color='grey', linestyle=':')
    ax[i, j].xaxis.set_major_locator(MonthLocator())
    ax[i, j].xaxis.set_major_formatter(ticker.NullFormatter())
    ax[i, j].yaxis.set_major_formatter(ticker.PercentFormatter(1, decimals=2))

#save_chart('stopy zwrotu')

In [ ]:
pairs_2 = np.array([
    [0, 1, 0],
    [0, 2, 1],
    [1, 2, 2]
])
fig, ax = plt.subplots(1, 3, figsize=(15,5), sharex=True, sharey=True)
for i, j, k in pairs_2:
    ax[k].scatter(sr.iloc[:, i], sr.iloc[:, j], s=30, marker='D', c="#9C9C9C", alpha=.9, edgecolor='black', linewidth=.4, zorder=50)
    lim = ax[k].get_xlim()
    ax[k].plot(lim, lim, linestyle='--', alpha=.6, c="#1B1B1B", zorder=60, linewidth=1)
    ax[k].set_ylim(lim)
    ax[k].set_xlim(lim)
    ax[k].axhline(c='black', linewidth=.4, linestyle=':', zorder=20)
    ax[k].axvline(c='black', linewidth=.4, linestyle=':', zorder=20)
    ax[k].grid(True, alpha=.6, zorder=0)
    ax[k].set_xlabel(sr.iloc[:, i].name, fontsize=15)
    ax[k].set_ylabel(sr.iloc[:, j].name, fontsize=15)

save_chart('rozrzut')